# Dim_Time Generation Notebook

This notebook generates a comprehensive time dimension table with:
- Time-of-day attributes (hour, minute, second)
- 12/24 hour formats
- Business hour indicators
- Time period classifications

**Prerequisites:**
1. Lakehouse attached to this notebook
2. Target Fabric Warehouse configured

## 1. Configuration & Setup

In [ ]:
# CONFIGURATION
# Get workspace and lakehouse IDs dynamically (for reference)
workspace_id = notebookutils.runtime.context.get('currentWorkspaceId', '')
lakehouse_id = notebookutils.runtime.context.get('defaultLakehouseId', '')

print(f"Workspace ID: {workspace_id}")
print(f"Lakehouse ID: {lakehouse_id}")

# TARGET WAREHOUSE CONFIGURATION
# Specify the name of your Fabric Warehouse where tables will be created
warehouse_name = "YourWarehouse"  # UPDATE THIS: Your Fabric Warehouse name

# Time granularity: 'second' or 'minute'
# 'second' generates 86,400 rows (every second of the day)
# 'minute' generates 1,440 rows (every minute of the day)
time_granularity = "minute"  # Recommended: 'minute' for most use cases

print(f"Target Warehouse: {warehouse_name}")
print(f"Time granularity: {time_granularity}")

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, concat, lpad, floor, expr
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
from datetime import time, timedelta

print("Libraries imported successfully")

## 2. Table Creation (DDL)

In [ ]:
# Create Dim_Time table
print(f"Setting database context to warehouse: {warehouse_name}")
spark.sql(f"USE `{warehouse_name}`")

ddl_sql = """
DROP TABLE IF EXISTS Dim_Time;

CREATE TABLE Dim_Time (
    -- Primary Key
    TimeKey INT NOT NULL,
    Time VARCHAR(8) NOT NULL,
    
    -- Time Components
    Hour24 INT,
    Hour12 INT,
    Minute INT,
    Second INT,
    
    -- Formatted Time
    AMPM VARCHAR(2),
    TimeDisplay12 VARCHAR(11),
    TimeDisplay24 VARCHAR(8),
    HourMinute VARCHAR(5),
    
    -- Time Calculations
    MinuteOfDay INT,
    SecondOfDay INT,
    
    -- Time Categories
    IsBusinessHour BOOLEAN,
    DayPeriod VARCHAR(20),
    HourBucket VARCHAR(20),
    QuarterHour INT,
    HalfHour INT,
    
    -- Metadata
    LoadDate TIMESTAMP
);
"""

print("Executing DDL in warehouse...")
for statement in ddl_sql.split(';'):
    if statement.strip():
        spark.sql(statement)

print(f"Table Dim_Time created successfully in warehouse: {warehouse_name}")

## 3. Data Generation

In [ ]:
# Generate time sequence based on granularity
if time_granularity == "second":
    # Generate every second (0 to 86399)
    max_seconds = 86400
    time_df = spark.range(0, max_seconds).toDF("SecondOfDay")
    print(f"Generating {max_seconds} time records (second granularity)")
else:
    # Generate every minute (0 to 1439)
    max_minutes = 1440
    time_df = spark.range(0, max_minutes).toDF("MinuteOfDay")
    time_df = time_df.withColumn("SecondOfDay", col("MinuteOfDay") * 60)
    print(f"Generating {max_minutes} time records (minute granularity)")

print(f"Initial records: {time_df.count()}")
time_df.show(5)

In [ ]:
# Calculate time components
dim_time_df = time_df \
    .withColumn("Hour24", floor(col("SecondOfDay") / 3600).cast("int")) \
    .withColumn("Minute", floor((col("SecondOfDay") % 3600) / 60).cast("int")) \
    .withColumn("Second", (col("SecondOfDay") % 60).cast("int"))

# Calculate MinuteOfDay if not already present
if time_granularity == "second":
    dim_time_df = dim_time_df.withColumn(
        "MinuteOfDay",
        (col("Hour24") * 60 + col("Minute")).cast("int")
    )

# 12-hour format
dim_time_df = dim_time_df \
    .withColumn("Hour12", 
                when(col("Hour24") == 0, 12)
                .when(col("Hour24") > 12, col("Hour24") - 12)
                .otherwise(col("Hour24"))) \
    .withColumn("AMPM",
                when(col("Hour24") < 12, "AM")
                .otherwise("PM"))

print("Time components calculated")
dim_time_df.show(5)

In [ ]:
# Create formatted time strings
dim_time_df = dim_time_df \
    .withColumn("TimeDisplay24",
                concat(
                    lpad(col("Hour24").cast("string"), 2, "0"),
                    lit(":"),
                    lpad(col("Minute").cast("string"), 2, "0"),
                    lit(":"),
                    lpad(col("Second").cast("string"), 2, "0")
                )) \
    .withColumn("HourMinute",
                concat(
                    lpad(col("Hour24").cast("string"), 2, "0"),
                    lit(":"),
                    lpad(col("Minute").cast("string"), 2, "0")
                )) \
    .withColumn("TimeDisplay12",
                concat(
                    lpad(col("Hour12").cast("string"), 2, "0"),
                    lit(":"),
                    lpad(col("Minute").cast("string"), 2, "0"),
                    lit(":"),
                    lpad(col("Second").cast("string"), 2, "0"),
                    lit(" "),
                    col("AMPM")
                ))

# Time is same as TimeDisplay24 for consistency
dim_time_df = dim_time_df.withColumn("Time", col("TimeDisplay24"))

# TimeKey in HHMMSS format
dim_time_df = dim_time_df.withColumn(
    "TimeKey",
    (col("Hour24") * 10000 + col("Minute") * 100 + col("Second")).cast("int")
)

print("Formatted time strings created")
dim_time_df.select("TimeKey", "Time", "TimeDisplay12", "TimeDisplay24").show(5)

In [ ]:
# Calculate time categories

# Business hours (8 AM to 5 PM)
dim_time_df = dim_time_df.withColumn(
    "IsBusinessHour",
    when((col("Hour24") >= 8) & (col("Hour24") < 17), True).otherwise(False)
)

# Day period classification
dim_time_df = dim_time_df.withColumn(
    "DayPeriod",
    when(col("Hour24") < 6, "Night")
    .when(col("Hour24") < 12, "Morning")
    .when(col("Hour24") < 17, "Afternoon")
    .when(col("Hour24") < 21, "Evening")
    .otherwise("Night")
)

# Hour bucket (e.g., "08:00-09:00")
dim_time_df = dim_time_df.withColumn(
    "HourBucket",
    concat(
        lpad(col("Hour24").cast("string"), 2, "0"),
        lit(":00-"),
        lpad((col("Hour24") + 1).cast("string"), 2, "0"),
        lit(":00")
    )
)

# Quarter hour (1-4 for each hour)
dim_time_df = dim_time_df.withColumn(
    "QuarterHour",
    floor(col("Minute") / 15).cast("int") + 1
)

# Half hour (1-2 for each hour)
dim_time_df = dim_time_df.withColumn(
    "HalfHour",
    floor(col("Minute") / 30).cast("int") + 1
)

print("Time categories calculated")
dim_time_df.select(
    "Time", "IsBusinessHour", "DayPeriod", "HourBucket", "QuarterHour"
).show(10)

## 4. Final Data Preparation

In [ ]:
# Add load timestamp
from pyspark.sql.functions import current_timestamp

dim_time_df = dim_time_df.withColumn("LoadDate", current_timestamp())

# Select columns in correct order
dim_time_final = dim_time_df.select(
    "TimeKey",
    "Time",
    "Hour24",
    "Hour12",
    "Minute",
    "Second",
    "AMPM",
    "TimeDisplay12",
    "TimeDisplay24",
    "HourMinute",
    "MinuteOfDay",
    "SecondOfDay",
    "IsBusinessHour",
    "DayPeriod",
    "HourBucket",
    "QuarterHour",
    "HalfHour",
    "LoadDate"
)

print(f"Final dataset prepared")
print(f"Total rows: {dim_time_final.count()}")

## 5. Data Quality Checks

In [ ]:
# Quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check for duplicates
duplicate_count = dim_time_final.groupBy("TimeKey").count().filter(col("count") > 1).count()
print(f"\n1. Duplicate TimeKeys: {duplicate_count}")
if duplicate_count > 0:
    print("   ⚠️ WARNING: Found duplicate TimeKeys!")
else:
    print("   ✓ No duplicates found")

# Check time range
min_time = dim_time_final.agg({"TimeKey": "min"}).collect()[0][0]
max_time = dim_time_final.agg({"TimeKey": "max"}).collect()[0][0]
print(f"\n2. Time Range: {min_time} to {max_time}")

# Check expected row count
actual_count = dim_time_final.count()
if time_granularity == "second":
    expected_count = 86400
else:
    expected_count = 1440

print(f"\n3. Row Count:")
print(f"   Expected: {expected_count}")
print(f"   Actual: {actual_count}")
if expected_count == actual_count:
    print("   ✓ Row count matches expected")
else:
    print(f"   ⚠️ WARNING: Row count mismatch!")

# Business hours statistics
business_hour_count = dim_time_final.filter(col("IsBusinessHour") == True).count()
business_hour_pct = (business_hour_count / actual_count) * 100
print(f"\n4. Business Hours: {business_hour_count} records ({business_hour_pct:.1f}%)")

# Day period distribution
print(f"\n5. Day Period Distribution:")
dim_time_final.groupBy("DayPeriod").count().orderBy("DayPeriod").show()

# Sample data
print(f"\n6. Sample Time Records:")
dim_time_final.select(
    "TimeKey", "TimeDisplay24", "TimeDisplay12", 
    "IsBusinessHour", "DayPeriod"
).orderBy("TimeKey").show(10)

print("\n" + "=" * 60)
print("Quality checks completed")
print("=" * 60)

## 6. Load Data to Warehouse

In [ ]:
# Load data to warehouse table
print("Loading data to Dim_Time table...")

try:
    # Ensure we're using the warehouse database
    spark.sql(f"USE `{warehouse_name}`")
    
    # Write to table (overwrite mode)
    dim_time_final.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable("Dim_Time")
    
    print("✓ Data loaded successfully")
    
    # Verify load
    row_count = spark.sql("SELECT COUNT(*) as cnt FROM Dim_Time").collect()[0]['cnt']
    print(f"✓ Verified: {row_count} rows in {warehouse_name}.Dim_Time table")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise

In [ ]:
# Display sample of loaded data
print("\nSample data from Dim_Time:")

# Ensure we're using the warehouse database
spark.sql(f"USE `{warehouse_name}`")

spark.sql("""
    SELECT 
        TimeKey,
        TimeDisplay24,
        TimeDisplay12,
        Hour24,
        Minute,
        IsBusinessHour,
        DayPeriod,
        HourBucket
    FROM Dim_Time
    WHERE Minute = 0
    ORDER BY TimeKey
    LIMIT 24
""").show(24, truncate=False)

## 7. Summary

**Dim_Time Generation Complete!**

The time dimension table has been successfully generated and loaded with:
- Time-of-day attributes (24h and 12h formats)
- Business hour indicators
- Day period classifications
- Time bucketing and calculations
- Data quality validation

**Usage Examples:**
```sql
-- Find all business hours
SELECT * FROM Dim_Time WHERE IsBusinessHour = 1;

-- Get morning time slots
SELECT * FROM Dim_Time WHERE DayPeriod = 'Morning';

-- Join with fact table
SELECT 
    f.*, 
    t.TimeDisplay12, 
    t.DayPeriod
FROM FactTable f
JOIN Dim_Time t ON f.TimeKey = t.TimeKey;
```